In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from datasets import load_dataset
from PIL import Image
import numpy as np
import wandb
from tqdm import tqdm

# For metrics & viz:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

Load Dataset

In [2]:
import os
os.environ["WANDB_API_KEY"] = "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"  

In [3]:
ds = load_dataset("HichTala/coco-background", streaming=True)

print(ds)
print(ds["train"].features)
print("num train (for streaming, this might not be precise):")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/103 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/103 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['image', 'label'],
        num_shards: 103
    })
    validation: IterableDataset({
        features: ['image', 'label'],
        num_shards: 5
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['airplane', 'apple', 'background', 'backpack', 'banana', 'baseball bat', 'baseball glove', 'bear', 'bed', 'bench', 'bicycle', 'bird', 'boat', 'book', 'bottle', 'bowl', 'broccoli', 'bus', 'cake', 'car', 'carrot', 'cat', 'cell phone', 'chair', 'clock', 'couch', 'cow', 'cup', 'dining table', 'dog', 'donut', 'elephant', 'fire hydrant', 'fork', 'frisbee', 'giraffe', 'hair drier', 'handbag', 'horse', 'hot dog', 'keyboard', 'kite', 'knife', 'laptop', 'microwave', 'motorcycle', 'mouse', 'orange', 'oven', 'parking meter', 'person', 'pizza', 'potted plant', 'refrigerator', 'remote', 'sandwich', 'scissors', 'sheep', 'sink', 'skateboard', 'skis', 'snowboard', 'spoon', 'sports ball', 'stop sign', 'suitcase', 'su

In [4]:
IMAGE_KEY = "image"
LABEL_KEY = "label"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Transforms
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



In [5]:
class HFImageIterableDataset(IterableDataset):
    def __init__(self, hf_split, transform=None, max_samples=None):
        """
        Args:
            hf_split: HuggingFace streaming dataset split
            transform: torchvision transforms
            max_samples: Maximum samples per epoch
        """
        self.hf_split = hf_split
        self.transform = transform
        self.max_samples = max_samples

    def __iter__(self):
        count = 0
        try:
            for ex in self.hf_split:
                # Check max samples limit
                if self.max_samples and count >= self.max_samples:
                    break

                try:
                    # Process image
                    img = ex[IMAGE_KEY]
                    if not isinstance(img, Image.Image):
                        img = Image.fromarray(np.array(img))
                    img = img.convert("RGB")

                    # Get label
                    y = ex[LABEL_KEY]

                    # Apply transforms
                    if self.transform:
                        img = self.transform(img)

                    count += 1
                    yield img, y

                except Exception as e:
                    print(f"Error processing sample {count}: {e}")
                    continue

        except Exception as e:
            print(f"Error in dataset iteration: {e}")
            raise

  # def get_num_classes(ds_split):
  #   """Extract number of classes from dataset features"""
  #   if LABEL_KEY in ds_split.features:
  #       feature = ds_split.features[LABEL_KEY]
  #       if hasattr(feature, 'num_classes'):
  #           return feature.num_classes
  #   return None

In [6]:
def build_loaders(ds, cfg):
    train_split = ds["train"].shuffle(seed=cfg["seed"], buffer_size=cfg["shuffle_buffer"])
    val_split = ds["validation"]

    train_samples = cfg["steps_per_epoch"] * cfg["batch_size"]
    val_samples = cfg["val_steps"] * cfg["batch_size"]

    train_ds = HFImageIterableDataset(train_split, transform=train_tfms, max_samples=train_samples)
    val_ds = HFImageIterableDataset(val_split, transform=val_tfms, max_samples=val_samples)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        num_workers=0,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["batch_size"],
        num_workers=0,
        pin_memory=True
    )
    return train_loader, val_loader

Config Wandb

In [7]:
config = {
    "batch_size": 32,
    "lr": 1e-4,
    "epochs": 5,
    "steps_per_epoch": 2000,   # define what an "epoch" means for streaming
    "val_steps": 300,
    "shuffle_buffer": 1000,
    "dataset": "HichTala/coco-background",
    "seed": 42
}

wandb.init(
    project="domain-shift-coco-dota",
    name="resnet50_finetuned_on_coco",
    config=config
)

# Optional: make W&B x-axis consistent and avoid “step vs step” plots
wandb.define_metric("global_step")
wandb.define_metric("train/*", step_metric="global_step")
wandb.define_metric("val/*", step_metric="global_step")
wandb.define_metric("epoch/*", step_metric="global_step")


wandb: Currently logged in as: mekaarikuchiri (mekaarikuchiri-imt-mines-al-s) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [8]:
num_classes = ds["train"].features["label"].num_classes
class_names = ds["train"].features["label"].names

print("num_classes:", num_classes)
print("first 5 classes:", class_names[:5])

train_loader, val_loader = build_loaders(ds, config)

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, num_classes)
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)

num_classes: 81
first 5 classes: ['airplane', 'apple', 'background', 'backpack', 'banana']
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


In [11]:
@torch.no_grad()
def validate(model, loader, criterion, cfg, global_step_end):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    progress_bar = tqdm(loader, total=cfg["val_steps"], desc="Validation", mininterval=2.0)
    for step, (x, y) in enumerate(progress_bar):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

        progress_bar.set_postfix(loss=running_loss / total, acc=correct / total)

        if step + 1 >= cfg["val_steps"]:
            break

    val_loss = running_loss / max(total, 1)
    val_acc = correct / max(total, 1)

    wandb.log({
        "global_step": global_step_end,
        "val/loss": val_loss,
        "val/acc": val_acc
    })

    return val_loss, val_acc


def train(model, loader, criterion, optimizer, cfg, epoch, global_step_start):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    progress_bar = tqdm(loader, total=cfg["steps_per_epoch"], desc=f"Epoch {epoch+1}/{cfg['epochs']}", mininterval=2.0)

    global_step = global_step_start

    for step, (x, y) in enumerate(progress_bar):
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

        progress_bar.set_postfix(loss=running_loss / total, acc=correct / total)

        if step % 50 == 0:
            wandb.log({
                "global_step": global_step,
                "train/step_loss": loss.item(),
                "train/step_acc": (preds == y).float().mean().item(),
            })

        global_step += 1

        if step + 1 >= cfg["steps_per_epoch"]:
            break

    train_loss = running_loss / max(total, 1)
    train_acc = correct / max(total, 1)

    wandb.log({
        "global_step": global_step,
        "epoch/idx": epoch + 1,
        "epoch/train_loss": train_loss,
        "epoch/train_acc": train_acc
    })

    return train_loss, train_acc, global_step


In [12]:
def save_checkpoint(model, optimizer, epoch, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, path)
    wandb.save(path)


global_step = 0
best_val_acc = 0.0  # Keep track of the best validation accuracy
print("Starting training...")

for epoch in range(config["epochs"]):
    train_loss, train_acc, global_step = train(
        model, train_loader, criterion, optimizer, config, epoch, global_step
    )

    val_loss, val_acc = validate(
        model, val_loader, criterion, config, global_step_end=global_step
    )

    print(
        f"\nEpoch {epoch+1}/{config['epochs']} Summary:\n"
        f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f}\n"
        f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.3f}\n"
    )

    # Save the best model based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"New best validation accuracy: {best_val_acc:.4f}. Saving best model...")
        save_checkpoint(model, optimizer, epoch, f"checkpoints/best_model.pth")

    save_checkpoint(model, optimizer, epoch, f"checkpoints/checkpoint_epoch_{epoch+1}.pth")

final_path = "checkpoints/resnet50_coco_background_final.pth"
torch.save(model.state_dict(), final_path)
wandb.save(final_path)

wandb.finish()
print("Training completed!")


Starting training...


Epoch 1/5:   0%|          | 0/2000 [00:00<?, ?it/s]

Validation: 100%|█████████▉| 299/300 [04:14<00:00,  1.18it/s, acc=0.00396, loss=13.8] 



Epoch 1/5 Summary:
  Train Loss: 0.1868 | Train Acc: 0.950
  Val   Loss: 13.8151 | Val   Acc: 0.004

New best validation accuracy: 0.0040. Saving best model...


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Validation: 100%|█████████▉| 299/300 [06:47<00:01,  1.36s/it, acc=0, loss=12.4]  


...

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Validation: 100%|█████████▉| 299/300 [06:25<00:01,  1.29s/it, acc=0.0132, loss=17.5] 



Epoch 5/5 Summary:
  Train Loss: 0.0891 | Train Acc: 0.974
  Val   Loss: 17.4776 | Val   Acc: 0.013

New best validation accuracy: 0.0132. Saving best model...


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


epoch/idx,▁▃▅▆█
epoch/train_acc,▁▅▅▇█
epoch/train_loss,█▄▃▂▁
global_step,▁▁▁▁▂▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█
train/step_acc,█▇████▁██▇███▅█████▇█████████████████▇█▅
train/step_loss,▅▂▁▁▁▁▁▁▁▁█▁▁▁▁▂▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▃▁▂▄█
val/loss,▃▁▅▆█
epoch/idx,5
epoch/train_acc,0.97441
epoch/train_loss,0.08906


Training completed!
